# 04. Preparación de datos — versión 2

## Objetivo

Reconstruir el dataset de modelado a partir de la salida limpia del notebook 03, aplicando una limpieza menos agresiva.

En esta versión:

- Se parte de `klosa_liver_stage1_special_codes_cleaned.csv`.
- Se eliminan variables vacías, constantes y con más del 50 % de valores ausentes.
- Se corrigen valores lógicamente imposibles convirtiéndolos en `NaN`.
- Se clasifican las variables en continuas y categóricas.
- Se auditan los outliers estadísticos, pero **no se eliminan variables por tener muchos outliers**.
- Se conservan los valores extremos plausibles para que XGBoost y CatBoost puedan explotarlos.
- No se imputan ni normalizan permanentemente los datos en este notebook.
- Se genera un dataset V2 para el entrenamiento posterior.

La imputación, normalización, One-Hot Encoding y balanceo se realizarán dentro de los pipelines de entrenamiento para evitar fuga de información.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
TARGET = "incident_liver_disease_10y"
ID_COLUMN = "pid"
WEIGHT_COLUMN = "wgt_c"

MAX_MISSING_PERCENTAGE = 50.0
OUTLIER_IQR_FACTOR = 1.5


In [2]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_PATH = (
    PROJECT_ROOT / "data" / "interim" /
    "klosa_liver_stage1_special_codes_cleaned.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "klosa_liver_modeling_dataset_v2.csv"
)

REPORTS_DIR = PROJECT_ROOT / "reports" / "preprocessing_v2"
CONFIGS_DIR = PROJECT_ROOT / "configs"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Entrada:  {INPUT_PATH}")
print(f"Salida:   {OUTPUT_PATH}")


Proyecto: C:\Users\DAVID\TFM_Liver_Disease_Risk
Entrada:  C:\Users\DAVID\TFM_Liver_Disease_Risk\data\interim\klosa_liver_stage1_special_codes_cleaned.csv
Salida:   C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_modeling_dataset_v2.csv


In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"No se ha encontrado el dataset de entrada:\n{INPUT_PATH}")

df = pd.read_csv(INPUT_PATH, low_memory=False)

print("Dataset cargado correctamente.")
print(f"Dimensiones: {df.shape}")
display(df.head())


Dataset cargado correctamente.
Dimensiones: (6560, 305)


,pid,a002_age,a002m,a002y,a003,a030,a032,a035_01,a035_02,a035_03,a035_04,a035_05,a035_06,addic,adl,agriculture,alc,annuity,assetinc,ba003,ba068,ba075,ba_resp,bb_adl1,bb_adl2,bb_adl3,bb_adl_num1,bb_adl_num2,bm3,bm4,bm5,bm6,bmi,bp1,bp1_1,bp1_2,bp3,bp4,bp5,bp6,businessfarm,c001,c003,c005,c007m,c007y,c012m,c012y,c017m,c017y,c024m,c024y,c034m,c034y,c039m,c039y,c044m,c044y,c049m,c049y,c056,c064m,c064y,c068,c081,c082,c085,c102,c105,c106,c107,c108,c109,c111,c112,c124m,c124y,c126,c127,c128,c129,c130,c131,c132,c133,c134,c135,c142,c143,c144,c145,c146,c147,c148,c149,c150,c151,c152,c201,c202,c203,c204,c205,c206,c207,c208,c209,c210,c211,c212,c213,c214,c215,c216,c217,c301,c302,c303,c304,c305,c306,c309,c310,c311,c312,c313,c318,c330,c333,c334,c337,c340,c343,c346,c550,chronic_a,chronic_b,chronic_c,chronic_d,chronic_f,chronic_g,chronic_h,chronic_i,chronic_j,chronic_sum,contact1,contact2,d_com001,d_com014,d_com015,...,d_com049,d_com052,d_com053,d_com054,d_com055,d_com056,d_com057,d_com060,d_com066,d_com068,d_com073,d_com074,d_com075,d_com076,d_com077,d_com078,d_com079,d_com080,d_com081,d_com082,d_com084,d_com085,d_com089,d_com090,d_com092,d_com093,d_com095,d_com096,d_com097,d_com098,d_com101,d_com102,d_com104,d_com105,d_com106,d_com107,d_com108,d_com110,d_com118,d_com125,d_com130,d_com137,d_com138,d_com139m,d_com139y,d_com140,d_com142,d_com144,d_com145,d_com149,d_com157,ea_resp,earned,ecoact_s,edu,edu_s,emp,enu_type,exploit,f001type,f013_n,fam1,financial,financialasset,fromchildren,fromothers,fromparent,full_parttime,g003,g004,g007,g008,g009,g010,g011,g012,g026,g027,g028,g029,g030,gender1,guarantee,hhinc,hhsize,house_sum,iadl,ind,industrial,insurance,job,job_search,job_startm,job_starty,l_sum,labor_st,livenear,livewith,livewithnm,marital,mgrip,mniw_d,mniw_m,n_days,national,oopm1,oopm2,other,otherasset,panel_n,particular,passets,pensionins,personal,pinc,pliabilities,pnetassets,present_ecotype,present_labor,publictrans,realestate,realestateinc,region1,region2,region3,regularins,residence,residence_,retired,s_sum,selfemployed,sideline,size_c,size_n,smoke,socialsecurity,socialwelfare,status,target1,tochildren,toothers,toparent,transferfrom,transferto,unemployment,wage,wgt_c,wholelifeins,year2,incident_liver_disease_10y
0,11,73,4,1933,1,2,10,3.0000,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,3,"1,200.0000",NaN,4,6.0000,5.0000,1,5,5,5,NaN,NaN,NaN,NaN,NaN,NaN,25.9695,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,5,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,000.0000",1,NaN,NaN,5.0000,5.0000,1,5,1.0000,60.0000,5,152.0000,5,4.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,1.0000,1.0000,1.0000,4.0000,1.0000,1.0000,4.0000,2.0000,1.0000,4,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1.0000,5.0000,NaN,1.0000,1.0000,12.0000,NaN,5.0000,NaN,NaN,1,0,0.0000,0,0,12.0000,0.0000,5,5,1,5,5,5,5,5,5,5,1,NaN,1,NaN,1.0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,-6.0000,2.0000,-6.0000,NaN,1,NaN,1.0000,NaN,1,NaN,NaN,50.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0000,NaN,NaN,NaN,50,50,NaN,70.0000,70,5,NaN,"1,200.0000",1,NaN,0,NaN,NaN,NaN,NaN,5.0000,NaN,NaN,NaN,5.0000,NaN,NaN,NaN,4,17.7500,11,10,NaN,NaN,NaN,30.0000,NaN,NaN,1,"1,200.0000","50,000.0000",NaN,NaN,"1,250.0000",NaN,"50,000.0000",3,5,"1,200.0000","50,000.0000",NaN,11,1,1,NaN,"50,000.0000",1.0000,0.0000,3.0000,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0000,NaN,NaN,NaN,"1,218.3210",NaN,-6.0000,0
1,21,51,3,1955,1,1,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,3,NaN,NaN,2,0.0000,NaN,1,5,5,5,NaN,NaN,NaN,NaN,NaN,NaN,23.6340,2,5.0000,NaN,5.0000,4.0000,6.0000,8.0000,NaN,2,5,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,1.0000,5.0000,5,5,NaN,59.0000,5,158.0000,5,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,1.0000,1.0000,1.0000,4.0000,1.0000,1.0000,4.0000,1.0000,1.0000,2,1,1,1,1,1,1,1,1,1,1,1,

In [4]:
required_columns = {ID_COLUMN, TARGET}
missing_required = required_columns - set(df.columns)

if missing_required:
    raise KeyError(f"Faltan columnas obligatorias: {missing_required}")

invalid_target_values = set(df[TARGET].dropna().unique()) - {0, 1}
if invalid_target_values:
    raise ValueError(
        "La variable objetivo contiene valores no válidos: "
        f"{invalid_target_values}"
    )

numeric_columns_initial = df.select_dtypes(include=np.number).columns.tolist()
remaining_special_codes = int(
    df[numeric_columns_initial].isin([-8, -9]).sum().sum()
)

print("VALIDACIÓN INICIAL")
print("=" * 60)
print(f"Pacientes: {len(df):,}")
print(f"Variables: {df.shape[1]:,}")
print(f"PID únicos: {df[ID_COLUMN].nunique(dropna=True):,}")
print(f"PID duplicados: {df[ID_COLUMN].duplicated().sum():,}")
print(f"Nulos en target: {df[TARGET].isna().sum():,}")
print(f"Códigos -8/-9 restantes: {remaining_special_codes:,}")
print(f"Casos positivos: {int(df[TARGET].sum()):,}")
print(f"Prevalencia positiva: {df[TARGET].mean():.2%}")

assert remaining_special_codes == 0


VALIDACIÓN INICIAL
Pacientes: 6,560
Variables: 305
PID únicos: 6,560
PID duplicados: 0
Nulos en target: 0
Códigos -8/-9 restantes: 0
Casos positivos: 131
Prevalencia positiva: 2.00%


## 1. Limpieza básica

Solo se eliminan aquí registros o columnas que claramente no aportan información:

- Filas sin variable objetivo.
- Duplicados completos.
- Variables completamente vacías.
- Variables constantes.

Los outliers estadísticos **no se eliminan**.


In [5]:
df_clean = df.copy(deep=True)
initial_shape = df_clean.shape

# Eliminar filas sin objetivo
df_clean = df_clean.loc[df_clean[TARGET].notna()].copy()
df_clean[TARGET] = pd.to_numeric(df_clean[TARGET], errors="raise").astype("int8")

# Eliminar duplicados completos
df_clean = df_clean.drop_duplicates().copy()

protected_columns = {ID_COLUMN, TARGET, WEIGHT_COLUMN}

all_missing_columns = [
    column for column in df_clean.columns
    if df_clean[column].notna().sum() == 0
    and column not in protected_columns
]

constant_columns = [
    column for column in df_clean.columns
    if df_clean[column].nunique(dropna=True) <= 1
    and column not in protected_columns
]

safe_drop_columns = sorted(set(all_missing_columns) | set(constant_columns))

df_clean = df_clean.drop(columns=safe_drop_columns, errors="ignore")

print(f"Dimensiones iniciales: {initial_shape}")
print(f"Variables vacías/constantes eliminadas: {len(safe_drop_columns)}")
print(f"Dimensiones actuales: {df_clean.shape}")


Dimensiones iniciales: (6560, 305)
Variables vacías/constantes eliminadas: 0
Dimensiones actuales: (6560, 305)


## 2. Valores ausentes

Se mantiene la regla de eliminar variables con más del 50 % de valores ausentes.

Esta decisión es independiente del análisis de outliers.


In [6]:
missing_report = pd.DataFrame({
    "nulos": df_clean.isna().sum(),
    "porcentaje_nulos": df_clean.isna().mean().mul(100),
    "valores_unicos": df_clean.nunique(dropna=True)
}).sort_values("porcentaje_nulos", ascending=False)

display(missing_report.head(40))


,nulos,porcentaje_nulos,valores_unicos
industrial,6556,99.9390,3
businessfarm,6555,99.9238,5
unemployment,6549,99.8323,9
personal,6545,99.7713,15
d_com108,6540,99.6951,6
size_c,6536,99.6341,9
d_com102,6526,99.4817,16
d_com101,6525,99.4665,17
g012,6525,99.4665,9
d_com098,6519,99.3750,2


In [7]:
high_missing_columns = [
    column for column in df_clean.columns
    if df_clean[column].isna().mean() * 100 > MAX_MISSING_PERCENTAGE
    and column not in protected_columns
]

missing_exclusion_report = pd.DataFrame({
    "variable": high_missing_columns,
    "porcentaje_nulos": [
        df_clean[column].isna().mean() * 100
        for column in high_missing_columns
    ],
    "valores_unicos": [
        df_clean[column].nunique(dropna=True)
        for column in high_missing_columns
    ],
    "motivo": "Más del 50% de valores ausentes"
}).sort_values("porcentaje_nulos", ascending=False)

print(
    f"Variables con más del {MAX_MISSING_PERCENTAGE:.0f}% "
    f"de valores ausentes: {len(high_missing_columns)}"
)

display(missing_exclusion_report)

df_clean = df_clean.drop(columns=high_missing_columns, errors="ignore")

print(
    "Dimensiones después de eliminar variables con muchos nulos: "
    f"{df_clean.shape}"
)


Variables con más del 50% de valores ausentes: 177


,variable,porcentaje_nulos,valores_unicos,motivo
139,industrial,99.9390,3,Más del 50% de valores ausentes
21,businessfarm,99.9238,5,Más del 50% de valores ausentes
174,unemployment,99.8323,9,Más del 50% de valores ausentes
154,personal,99.7713,15,Más del 50% de valores ausentes
104,d_com108,99.6951,6,Más del 50% de valores ausentes
163,size_c,99.6341,9,Más del 50% de valores ausentes
99,d_com102,99.4817,16,Más del 50% de valores ausentes
135,g012,99.4665,9,Más del 50% de valores ausentes
98,d_com101,99.4665,17,Más del 50% de valores ausentes
97,d_com098,99.3750,2,Más del 50% de valores ausentes


Dimensiones después de eliminar variables con muchos nulos: (6560, 128)


## 3. Validación de rangos lógicos

Los valores imposibles se convierten en `NaN`, pero no se elimina la fila completa.

Se revisan de forma explícita:

- Edades: deben ser mayores que 0 y menores o iguales que 120.
- Porcentajes: deben estar entre 0 y 100.

La lista de porcentajes debe contener únicamente variables confirmadas como porcentuales.


In [8]:
AGE_COLUMNS = [
    column for column in df_clean.columns
    if (
        column.lower() == "age"
        or column.lower().endswith("_age")
        or column.lower().startswith("age_")
    )
]

if "a002_age" in df_clean.columns:
    AGE_COLUMNS = sorted(set(AGE_COLUMNS + ["a002_age"]))

# Añadir aquí SOLO variables confirmadas como porcentajes.
PERCENTAGE_COLUMNS = [
    # "nombre_variable"
]

print("Variables de edad detectadas:")
print(AGE_COLUMNS)
print("\nVariables porcentuales confirmadas:")
print(PERCENTAGE_COLUMNS)


Variables de edad detectadas:
['a002_age']

Variables porcentuales confirmadas:
[]


In [9]:
invalid_value_records = []

# Edades
for column in AGE_COLUMNS:
    if column not in df_clean.columns:
        continue

    series = pd.to_numeric(df_clean[column], errors="coerce")
    invalid_mask = (series <= 0) | (series > 120)
    invalid_count = int(invalid_mask.sum())

    if invalid_count > 0:
        invalid_value_records.append({
            "variable": column,
            "tipo": "edad",
            "regla": "0 < edad <= 120",
            "valores_invalidos": invalid_count,
            "porcentaje_invalidos": invalid_count / len(df_clean) * 100,
            "minimo_original": series.min(),
            "maximo_original": series.max()
        })
        df_clean.loc[invalid_mask, column] = np.nan

# Porcentajes
for column in PERCENTAGE_COLUMNS:
    if column not in df_clean.columns:
        raise KeyError(f"No existe la variable porcentual '{column}'.")

    series = pd.to_numeric(df_clean[column], errors="coerce")
    invalid_mask = (series < 0) | (series > 100)
    invalid_count = int(invalid_mask.sum())

    if invalid_count > 0:
        invalid_value_records.append({
            "variable": column,
            "tipo": "porcentaje",
            "regla": "0 <= porcentaje <= 100",
            "valores_invalidos": invalid_count,
            "porcentaje_invalidos": invalid_count / len(df_clean) * 100,
            "minimo_original": series.min(),
            "maximo_original": series.max()
        })
        df_clean.loc[invalid_mask, column] = np.nan

invalid_values_report = pd.DataFrame(invalid_value_records)
display(invalid_values_report)


""


## 4. Clasificación de variables

Se realiza una clasificación inicial:

- Variables no numéricas → categóricas.
- Variables numéricas con 10 o menos valores únicos → categóricas candidatas.
- El resto → continuas.

Las listas manuales permiten corregir casos concretos sin modificar el resto del pipeline.


In [10]:
feature_columns = [
    column for column in df_clean.columns
    if column not in {ID_COLUMN, TARGET, WEIGHT_COLUMN}
]

numeric_columns = df_clean[feature_columns].select_dtypes(include=np.number).columns.tolist()

non_numeric_columns = [
    column for column in feature_columns
    if column not in numeric_columns
]

# Correcciones manuales si son necesarias
MANUAL_CATEGORICAL_COLUMNS = [
    # "nombre_variable"
]

MANUAL_CONTINUOUS_COLUMNS = list(set(AGE_COLUMNS + PERCENTAGE_COLUMNS))

automatic_categorical_columns = [
    column for column in numeric_columns
    if df_clean[column].nunique(dropna=True) <= 10
    and column not in MANUAL_CONTINUOUS_COLUMNS
]

categorical_columns = sorted(
    (
        set(non_numeric_columns)
        | set(automatic_categorical_columns)
        | set(MANUAL_CATEGORICAL_COLUMNS)
    )
    - set(MANUAL_CONTINUOUS_COLUMNS)
)

continuous_columns = sorted(set(feature_columns) - set(categorical_columns))

print(f"Predictores totales: {len(feature_columns)}")
print(f"Variables continuas: {len(continuous_columns)}")
print(f"Variables categóricas: {len(categorical_columns)}")


Predictores totales: 125
Variables continuas: 30
Variables categóricas: 95


In [11]:
variable_type_report = pd.DataFrame({
    "variable": feature_columns,
    "tipo": [
        "continua" if column in continuous_columns else "categórica"
        for column in feature_columns
    ],
    "tipo_pandas": [str(df_clean[column].dtype) for column in feature_columns],
    "valores_unicos": [
        df_clean[column].nunique(dropna=True)
        for column in feature_columns
    ],
    "porcentaje_nulos": [
        df_clean[column].isna().mean() * 100
        for column in feature_columns
    ]
}).sort_values(["tipo", "variable"])

display(variable_type_report)


,variable,tipo,tipo_pandas,valores_unicos,porcentaje_nulos
3,a003,categórica,int64,2,0.0000
4,a030,categórica,int64,6,0.0000
5,a032,categórica,int64,10,0.0000
6,a035_02,categórica,float64,10,41.2043
7,adl,categórica,int64,8,0.0000
8,alc,categórica,int64,3,0.0000
9,ba003,categórica,int64,10,0.0000
11,ba075,categórica,float64,2,42.9268
12,ba_resp,categórica,int64,2,0.0000
13,bb_adl1,categórica,int64,2,0.0000


## 5. Auditoría de outliers

Se utiliza la regla del IQR para detectar valores extremos en variables continuas.

**Importante:** en esta versión los outliers se auditan, pero no se eliminan ni se eliminan variables por tener un porcentaje elevado de outliers.

Esto evita descartar variables económicas, clínicas o de comportamiento cuya distribución sea legítimamente asimétrica.


In [12]:
outlier_records = []

for column in continuous_columns:
    series = pd.to_numeric(df_clean[column], errors="coerce").dropna()

    if len(series) < 20:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        lower_limit = q1
        upper_limit = q3
        outlier_count = 0
    else:
        lower_limit = q1 - OUTLIER_IQR_FACTOR * iqr
        upper_limit = q3 + OUTLIER_IQR_FACTOR * iqr
        outlier_count = int(
            ((series < lower_limit) | (series > upper_limit)).sum()
        )

    outlier_percentage = outlier_count / len(series) * 100

    outlier_records.append({
        "variable": column,
        "observaciones_validas": len(series),
        "minimo": series.min(),
        "q1": q1,
        "mediana": series.median(),
        "q3": q3,
        "maximo": series.max(),
        "limite_inferior_iqr": lower_limit,
        "limite_superior_iqr": upper_limit,
        "numero_outliers": outlier_count,
        "porcentaje_outliers": outlier_percentage,
        "asimetria": series.skew()
    })

outlier_report = (
    pd.DataFrame(outlier_records)
    .sort_values("porcentaje_outliers", ascending=False)
    .reset_index(drop=True)
)

display(outlier_report.head(50))


,variable,observaciones_validas,minimo,q1,mediana,q3,maximo,limite_inferior_iqr,limite_superior_iqr,numero_outliers,porcentaje_outliers,asimetria
0,c330,6558,0.0000,0.0000,0.0000,1.0000,310.0000,-1.5000,2.5000,1285,19.5944,18.3922
1,c334,6560,0.0000,0.0000,0.0000,1.0000,300.0000,-1.5000,2.5000,1141,17.3933,10.8353
2,year2,6521,-6.0000,"1,934.0000","1,945.0000","1,954.0000","1,982.0000","1,904.0000","1,984.0000",1085,16.6386,-1.7929
3,oopm2,5071,0.0000,3.0000,18.0000,53.0000,"9,997.0000",-72.0000,128.0000,551,10.8657,26.6650
4,pnetassets,4425,"-96,700.0000",500.0000,"3,200.0000","10,120.0000","365,500.0000","-13,930.0000","24,550.0000",412,9.3107,6.7536
5,passets,4317,10.0000,992.0000,"4,300.0000","11,920.0000","365,500.0000","-15,400.0000","28,312.0000",382,8.8487,6.7354
6,c337,6554,0.0000,0.0000,2.0000,6.0000,360.0000,-9.0000,15.0000,518,7.9036,9.5488
7,pinc,5008,2.0000,152.0000,686.0000,"1,825.0000","48,000.0000","-2,357.5000","4,334.5000",297,5.9305,6.6813
8,g029,6412,0.0000,70.0000,80.0000,90.0000,100.0000,40.0000,120.0000,266,4.1485,-1.1314
9,g026,6560,0.0000,50.0000,60.0000,80.0000,100.0000,5.0000,125.0000,234,3.5671,-0.5692


In [13]:
high_outlier_variables = (
    outlier_report
    .loc[
        outlier_report["porcentaje_outliers"] > 50,
        "variable"
    ]
    .tolist()
)

print(
    "Variables con más del 50% de valores marcados como outliers "
    f"por IQR: {len(high_outlier_variables)}"
)
print("Estas variables se CONSERVAN en la versión 2.")

display(
    outlier_report.loc[
        outlier_report["variable"].isin(high_outlier_variables)
    ]
)


Variables con más del 50% de valores marcados como outliers por IQR: 0
Estas variables se CONSERVAN en la versión 2.


,variable,observaciones_validas,minimo,q1,mediana,q3,maximo,limite_inferior_iqr,limite_superior_iqr,numero_outliers,porcentaje_outliers,asimetria


## 6. Resumen final de valores ausentes

Los valores ausentes que permanecen no se imputan aquí.

La imputación se realizará dentro de los pipelines del modelo, ajustándose únicamente con los datos de entrenamiento.


In [14]:
final_missing_report = pd.DataFrame({
    "nulos": df_clean.isna().sum(),
    "porcentaje_nulos": df_clean.isna().mean().mul(100),
    "valores_unicos": df_clean.nunique(dropna=True)
}).sort_values("porcentaje_nulos", ascending=False)

display(final_missing_report.head(40))


,nulos,porcentaje_nulos,valores_unicos
labor_st,3109,47.3933,3
retired,3109,47.3933,3
livewith,2952,45.0000,5
job_search,2832,43.1707,2
contact2,2828,43.1098,8
ba075,2816,42.9268,2
c068,2762,42.1037,2
c303,2761,42.0884,2
a035_02,2703,41.2043,10
c109,2595,39.5579,5


## 7. Guardado del dataset V2

El dataset se guarda con:

- Variables originales válidas.
- `NaN` conservados.
- Outliers plausibles conservados.
- Sin imputación.
- Sin normalización.
- Sin One-Hot Encoding.

Estas transformaciones se realizarán durante el entrenamiento.


In [15]:
removed_variables_report = pd.DataFrame({
    "variable": safe_drop_columns + high_missing_columns,
    "motivo": (
        ["Variable vacía o constante" for _ in safe_drop_columns]
        + ["Más del 50% de valores ausentes" for _ in high_missing_columns]
    )
})

variable_groups_v2 = {
    "continuous_columns": continuous_columns,
    "categorical_columns": categorical_columns,
    "percentage_columns": PERCENTAGE_COLUMNS,
    "age_columns": AGE_COLUMNS,
    "removed_empty_or_constant_columns": safe_drop_columns,
    "removed_high_missing_columns": high_missing_columns,
    "high_outlier_variables_retained": high_outlier_variables
}

df_clean.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

with open(
    CONFIGS_DIR / "variable_groups_v2.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        variable_groups_v2,
        file,
        indent=4,
        ensure_ascii=False
    )

missing_report.to_csv(
    REPORTS_DIR / "missing_values_before_filter.csv",
    encoding="utf-8-sig"
)

final_missing_report.to_csv(
    REPORTS_DIR / "missing_values_final.csv",
    encoding="utf-8-sig"
)

missing_exclusion_report.to_csv(
    REPORTS_DIR / "removed_high_missing_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

removed_variables_report.to_csv(
    REPORTS_DIR / "removed_variables.csv",
    index=False,
    encoding="utf-8-sig"
)

variable_type_report.to_csv(
    REPORTS_DIR / "variable_types.csv",
    index=False,
    encoding="utf-8-sig"
)

outlier_report.to_csv(
    REPORTS_DIR / "outlier_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

invalid_values_report.to_csv(
    REPORTS_DIR / "invalid_values.csv",
    index=False,
    encoding="utf-8-sig"
)

print("ARCHIVOS GUARDADOS")
print("=" * 60)
print(f"Dataset V2: {OUTPUT_PATH}")
print(f"Configuración: {CONFIGS_DIR / 'variable_groups_v2.json'}")
print(f"Informes: {REPORTS_DIR}")


ARCHIVOS GUARDADOS
Dataset V2: C:\Users\DAVID\TFM_Liver_Disease_Risk\data\processed\klosa_liver_modeling_dataset_v2.csv
Configuración: C:\Users\DAVID\TFM_Liver_Disease_Risk\configs\variable_groups_v2.json
Informes: C:\Users\DAVID\TFM_Liver_Disease_Risk\reports\preprocessing_v2


In [16]:
print("RESUMEN NOTEBOOK 04 V2")
print("=" * 70)
print(f"Dimensiones iniciales: {initial_shape}")
print(f"Dimensiones finales: {df_clean.shape}")
print(f"Variables vacías/constantes eliminadas: {len(safe_drop_columns)}")
print(f"Variables eliminadas por >50% de missing: {len(high_missing_columns)}")
print("Variables eliminadas por outliers: 0")
print(
    f"Variables con >50% de outliers conservadas: "
    f"{len(high_outlier_variables)}"
)
print(f"Variables continuas finales: {len(continuous_columns)}")
print(f"Variables categóricas finales: {len(categorical_columns)}")
print(f"Casos positivos: {int(df_clean[TARGET].sum())}")
print(f"Prevalencia positiva: {df_clean[TARGET].mean():.2%}")
print("\nDataset V2 listo para comparar XGBoost y CatBoost.")


RESUMEN NOTEBOOK 04 V2
Dimensiones iniciales: (6560, 305)
Dimensiones finales: (6560, 128)
Variables vacías/constantes eliminadas: 0
Variables eliminadas por >50% de missing: 177
Variables eliminadas por outliers: 0
Variables con >50% de outliers conservadas: 0
Variables continuas finales: 30
Variables categóricas finales: 95
Casos positivos: 131
Prevalencia positiva: 2.00%

Dataset V2 listo para comparar XGBoost y CatBoost.
